In [22]:
import os
from getpass import getpass

from google import genai
from google.genai import types
from ddgs import DDGS

print("Libraries imported successfully!")

Libraries imported successfully!


In [29]:
# Enter new Gemini API key
api_key = getpass("Enter your Gemini API Key: ")

# Connect Gemini
client = genai.Client(api_key=api_key)

# Select model
MODEL_NAME = "gemini-3.5-flash"

# Test connection
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say Hello, Travel Planner!"
)

print(response.text)

Enter your Gemini API Key: ··········
Hello! 🌍✈️ I am your Travel Planner, ready to help you map out your next adventure! 

Where are we dreaming of going today? Let me know if you have a specific destination in mind, a budget, a travel style (relaxing, adventurous, budget-friendly, luxury), or if you just need some inspiration!


In [37]:
from ddgs import DDGS  # Import library for internet searching

def search_destination(destination: str):  # Added : str type hint

    print("\n[Agent] Searching destination...")  # Display searching message

    try:  # Handle possible errors

        results = DDGS().text(  # Search the internet
            destination + " tourist attractions travel guide",
            max_results=5  # Get maximum 5 results
        )

        for result in results:  # Loop through search results

            print("\nTitle:", result.get("title", ""))  # Display title
            print("Information:", result.get("body", ""))  # Display description
            print("Website:", result.get("href", ""))  # Display website link

    except Exception as e:  # Catch errors
        print("Search unavailable:", e)  # Display error message


In [38]:
def calculate_budget(budget: float, days: int):  # Define function with budget and days

    print("\n[Agent] Calculating budget...")  # Display calculation message

    if budget <= 0 or days <= 0:  # Check whether inputs are valid
        return "Invalid budget or number of days."  # Return error message

    daily_budget = budget / days  # Calculate daily budget

    return f"""
    Total Budget: {budget}
    Total Days: {days}
    Daily Budget: {daily_budget:.2f}
    """  # Return budget details


In [39]:
def create_itinerary(destination: str, days: int):  # Define function

    print("\n[Agent] Creating itinerary...")  # Display message

    if days < 1 or days > 30:  # Validate number of days
        return "Please select between 1 and 30 days."  # Return error

    itinerary = []  # Create an empty list

    for day in range(1, days + 1):  # Loop through each travel day

        itinerary.append(
            f"Day {day}: Explore {destination}."  # Add daily plan
        )

    return "\n".join(itinerary)  # Combine all days into one string


In [40]:
def travel_agent(destination, days, budget, interests):

    question = f"""
    Create a travel plan using these details:

    Destination: {destination}
    Number of days: {days}
    Total budget: {budget}
    Interests: {interests}

    Your tasks:

    1. Search for tourist attractions.
    2. Calculate the daily budget.
    3. Create a day-wise itinerary.
    4. Suggest popular places to visit.
    5. Suggest local food.
    6. Suggest transportation options.
    7. Keep the plan within the given budget where possible.
    8. Mention that prices and opening hours must be verified.

    Give the final answer in simple English.
    """

    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=question,

        config=types.GenerateContentConfig(

            system_instruction="""
            You are an intelligent AI Travel Planner.

            Understand the user's requirements.

            Decide which tools are required.

            Use the search tool to find travel information.

            Use the budget tool to calculate expenses.

            Use the itinerary tool to prepare a travel schedule.

            Analyze the tool results and generate
            a complete travel plan.
            """,

            tools=[
                search_destination,
                calculate_budget,
                create_itinerary
            ],

            automatic_function_calling=
                types.AutomaticFunctionCallingConfig(
                    maximum_remote_calls=5
                )
        )
    )

    return response.text

In [41]:
print("======================================")
print("         AI TRAVEL PLANNER")
print("======================================")

destination = input("Enter destination: ")

days = int(input("Enter number of days: "))

budget = float(input("Enter total budget: "))

interests = input(
    "Enter your interests: "
)

print("\nTravel Agent is working...")

answer = travel_agent(
    destination,
    days,
    budget,
    interests
)

print("\n========== YOUR TRAVEL PLAN ==========")

print(answer)

         AI TRAVEL PLANNER
Enter destination: jaipur
Enter number of days: 3
Enter total budget: 67000
Enter your interests: historical place

Travel Agent is working...

[Agent] Searching destination...

Title: Jaipur Tourist Attractions & Travel Guide
Information: Travel Planning And Booking Made Easy. Plus, Free Cancellation Up To 24-Hours Before. Viator: Your Ticket to Unforgettable Experiences & Worry-Free Cancellations
Website: https://www.bing.com/aclick?ld=e8FbES3IFAeaPs_phJic7mYzVUCUw5N9vCt5eMlNhCyI-0v2HBoSSLf3nNDy_mJ0vVPs0JI9p_cdi1Do2lc2XCJenX4TMgyBsH2XhxEPOnTmTYKw0EmysiJNZwLCgaik2BVkyXEB1e9kMAne8uLbLEj6i4UVsMSidcG8G97blD_EQXnLbTDCsrQEGnDDFWNifVdqH4bGnYq1hkyvxKmGIoDttMcXQ&u=aHR0cHMlM2ElMmYlMmZhZC5kb3VibGVjbGljay5uZXQlMmZzZWFyY2hhZHMlMmZsaW5rJTJmY2xpY2slM2ZsaWQlM2QzOTcwMDA4Mzg2MDgwOTQ1MyUyNmRzX3Nfa3dnaWQlM2Q1ODcwMDAwOTA3NDQ2NzIzNSUyNmRzX2FfY2lkJTNkODI1Njg2NzU3NSUyNmRzX2FfY2FpZCUzZDIzODczOTQwNDU3JTI2ZHNfYV9hZ2lkJTNkMTk2OTMzMzg4MTM5JTI2ZHNfYV9saWQlM2Rkc2EtMjQxNDcyNDE0MzI3OCUyNiU